# 03 — Protect: DSPM → Zero Trust tier derivation

## Google Drive Setup

In [1]:
import os
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

# Check if drive is already mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
else:
  print("Drive is already mounted.")

print("Done!")

# Filepath Search
#search_term = "your_filename.ext"  # Change this
search_term = "synthetic_assets.csv"
!find /content/drive/MyDrive -maxdepth 15 -type f -iname "synthetic_assets.csv" -print


Mounted at /content/drive
Drive is already mounted.
Done!
/content/drive/MyDrive/data/synthetic_assets.csv


In [2]:
!pip -q install pandas numpy matplotlib  # safe to re-run

In [3]:
import pandas as pd
assets = pd.read_csv('/content/drive/MyDrive/data/synthetic_assets.csv')

WEIGHTS = {'sens': {'public':5,'internal':20,'confidential':45,'restricted':65}, 'internet':25,'pii':10,'phi':15,'enc_none':15,'enc_partial':7}
def risk(row):
    s = WEIGHTS['sens'][row['sensitivity']]
    if bool(row['internet_exposed']): s += WEIGHTS['internet']
    if bool(row['has_pii']): s += WEIGHTS['pii']
    if bool(row['has_phi']): s += WEIGHTS['phi']
    if row['encryption'] == 'none': s += WEIGHTS['enc_none']
    elif row['encryption'] in ('at_rest','in_transit'): s += WEIGHTS['enc_partial']
    return min(100, int(s))

def zt_tier(r):
    if r >= 85: return 'tier0_deny_by_default'
    if r >= 65: return 'tier1_strict_conditional'
    if r >= 45: return 'tier2_strong_auth'
    return 'tier3_baseline'

assets['risk'] = assets.apply(risk, axis=1)
assets['zt_tier'] = assets['risk'].apply(zt_tier)
assets[['asset_id','sensitivity','internet_exposed','has_pii','encryption','risk','zt_tier']].head(12)

,asset_id,sensitivity,internet_exposed,has_pii,encryption,risk,zt_tier
0,a000,confidential,False,True,at_rest,62,tier2_strong_auth
1,a001,confidential,False,True,both,55,tier2_strong_auth
2,a002,public,False,False,none,20,tier3_baseline
3,a003,internal,False,False,in_transit,27,tier3_baseline
4,a004,confidential,False,True,at_rest,62,tier2_strong_auth
5,a005,internal,False,False,none,35,tier3_baseline
6,a006,confidential,False,True,at_rest,62,tier2_strong_auth
7,a007,internal,False,False,at_rest,27,tier3_baseline
8,a008,internal,False,False,at_rest,27,tier3_baseline
9,a009,internal,False,False,in_transit,27,tier3_baseline


In [4]:
assets['zt_tier'].value_counts()

,count
zt_tier,
tier3_baseline,34
tier2_strong_auth,21
tier0_deny_by_default,15
tier1_strict_conditional,10
